# 5-Class Emergency Department Triage Classifier on FedMML Dataset (`models/train_fedmm_classifier.ipynb`)

This notebook trains **LightGBM Emergency Severity Index (ESI 1..5) triage models** on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)** using vital signs and core demographic features:

### 📊 Feature Roster & Target Specification
- **Required Predictor Features (6 Features)**:
  1. `age`: Patient age in years (numerical continuous).
  2. `sex`: Biological sex encoded as integer (`1 = Male ('M')`, `0 = Female ('F')`).
  3. `systolic_bp`: Systolic Blood Pressure in mmHg (numerical continuous).
  4. `heart_rate`: Heart Rate in beats per minute (bpm) (numerical continuous).
  5. `respiratory_rate`: Respiratory Rate in breaths per minute (numerical continuous).
  6. `spo2`: Blood Oxygen Saturation in percentage (%) (numerical continuous).
- **Target Class**:
  - `esi_level`: Emergency Severity Index triage score with 5 discrete levels (`1: Resuscitation`, `2: Emergent`, `3: Urgent`, `4: Less Urgent`, `5: Non-urgent`).
- **Data Cleaning Rule**:
  - Strictly drops any row containing at least a single `null` / `NaN` value across the 6 required features or the target column.

```mermaid
flowchart TD
    RawData["Raw Dataset (datasets/fedmml_ed_triage_dataset.csv)"] --> DropNA["Filter Complete Cases: Drop any NA in [age, sex, sbp, hr, rr, spo2, esi_level]"]
    DropNA --> Split["Stratified 3-Way Split: Train (80%), Val (10%), Test (10%)"]
    Split --> Preproc["StandardScaler Normalization"]
    
    Preproc --> ModelA["Model A: Direct 5-Class Multi-Class LightGBM (objective='multiclass')"]
    Preproc --> ModelB["Model B: Hierarchical Multi-Tier LightGBM Stacking (L1, L2, L3A, L3B + LogReg Meta)"]
    
    ModelA & ModelB --> Eval["Holdout Test Benchmark: Per-Class Breakdown, 5x5 Confusion Matrix, ROC-AUC Curves, Feature Importance"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Filter Nulls, Encode 'sex', & Stratified 3-Way Split
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading dataset from: {data_path}...")
raw_df = pd.read_csv(data_path)
initial_rows = len(raw_df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'
all_required_cols = required_features + [target_col]

print("=" * 80)
print(f"  INITIAL DATASET: {initial_rows:,} Total Rows, {raw_df.shape[1]} Columns")
print("=" * 80)
print("Null counts per required column before filtering:")
print(raw_df[all_required_cols].isnull().sum())
print("-" * 80)

# 2. Strictly Drop Rows With >= 1 Null in Required Features or Target
clean_df = raw_df.dropna(subset=all_required_cols).copy()
clean_rows = len(clean_df)
dropped_rows = initial_rows - clean_rows

print(f"✓ Dropped {dropped_rows:,} rows with missing values (Retained {clean_rows:,} complete cases, {clean_rows/initial_rows*100:.2f}%)")

# 3. Encode 'sex' Feature (M -> 1, F -> 0)
clean_df['sex_encoded'] = clean_df['sex'].astype(str).str.strip().str.upper().map({'M': 1, 'MALE': 1, 'F': 0, 'FEMALE': 0})
# Ensure any unexpected values are filled or filtered
clean_df = clean_df.dropna(subset=['sex_encoded']).copy()
clean_df['sex_encoded'] = clean_df['sex_encoded'].astype(int)
clean_df[target_col] = clean_df[target_col].astype(int)

feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

print("\nCleaned Dataset Target Distribution ('esi_level'):")
esi_dist = clean_df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} cases ({cnt/len(clean_df)*100:.2f}%)")
print("=" * 80)

# 4. Stratified 3-Way Partitioning (80% Train, 10% Validation, 10% Holdout Test)
X_all = clean_df[feature_names].values
y_all = clean_df[target_col].values

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Stratified Partition Complete:")
print(f"  * Train Set      : {len(X_train_raw):,} rows ({len(X_train_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} rows ({len(X_val_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} rows ({len(X_test_raw)/len(clean_df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Normalization (StandardScaler on Continuous Vitals)
# ---------------------------------------------------------
cont_indices = [0, 2, 3, 4, 5]  # age, systolic_bp, heart_rate, respiratory_rate, spo2

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_raw[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_raw[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_raw[:, cont_indices])

print(f"✓ Feature Matrices Normalized: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Model A — Direct 5-Class Multi-Class LightGBM Classifier
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING MODEL A: DIRECT 5-CLASS MULTI-CLASS LIGHTGBM CLASSIFIER")
print("=" * 80)

lgb_mc_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'class_weight': 'balanced',
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'min_child_samples': 20,
    'n_estimators': 200,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()
# Note: LightGBM multiclass expects 0-indexed classes: {1, 2, 3, 4, 5} -> {0, 1, 2, 3, 4}
model_mc_lgbm = lgb.LGBMClassifier(**lgb_mc_params)
model_mc_lgbm.fit(
    X_train, y_train - 1,
    eval_set=[(X_val, y_val - 1)],
    callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
)

print(f"✓ Direct 5-Class LightGBM trained in {time.time()-t0:.1f}s!")
probs_test_mc = model_mc_lgbm.predict_proba(X_test)
preds_test_mc = model_mc_lgbm.predict(X_test) + 1

In [ ]:
# ---------------------------------------------------------
# Step 4: Train Model B — Hierarchical Multi-Tier LightGBM Stacking Pipeline
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING MODEL B: HIERARCHICAL 4-TIER LIGHTGBM STACKING PIPELINE")
print("=" * 80)

def binary_numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]), dtype=np.float64)
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_bin_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42,
    'n_estimators': 150
}

t0 = time.time()

# 1. Layer 1: ESI 1 vs (ESI 2..5) with SMOTE
X_sm1, y_sm1 = binary_numpy_smote(X_train, (y_train == 1).astype(int))
l1_model = lgb.LGBMClassifier(**lgb_bin_params)
l1_model.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 2. Layer 2: ESI 2,3 vs ESI 4,5 on Non-ESI 1
m2_tr  = (y_train != 1); m2_val = (y_val != 1)
X_sm2, y_sm2 = binary_numpy_smote(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_model = lgb.LGBMClassifier(**lgb_bin_params)
l2_model.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 3. Layer 3A: ESI 2 vs ESI 3 on ESI 2,3
m3a_tr  = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
X_sm3a, y_sm3a = binary_numpy_smote(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_model = lgb.LGBMClassifier(**lgb_bin_params)
l3a_model.fit(X_sm3a, y_sm3a, eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 4. Layer 3B: ESI 4 vs ESI 5 on ESI 4,5
m3b_tr  = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
X_sm3b, y_sm3b = binary_numpy_smote(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_model = lgb.LGBMClassifier(**lgb_bin_params)
l3b_model.fit(X_sm3b, y_sm3b, eval_set=[(X_val[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Hierarchical Probability Computation
def compute_hierarchical_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

val_probs_stack  = compute_hierarchical_probs(l1_model, l2_model, l3a_model, l3b_model, X_val)
test_probs_stack = compute_hierarchical_probs(l1_model, l2_model, l3a_model, l3b_model, X_test)

# Calibrate Multinomial Logistic Regression Meta-Learner on Validation Probabilities
meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_learner.fit(val_probs_stack, y_val)

probs_test_stack = meta_learner.predict_proba(test_probs_stack)
preds_test_stack = meta_learner.predict(test_probs_stack)

print(f"✓ Hierarchical LightGBM Stacking Pipeline trained & calibrated in {time.time()-t0:.1f}s!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Holdout Test Set Benchmark & Comprehensive Per-Class Breakdown
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name, class_list=[1, 2, 3, 4, 5]):
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs = [], [], [], [], [], []
    for idx, cls in enumerate(class_list):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_mc    = get_per_class_breakdown(y_test, preds_test_mc, probs_test_mc, 'FedMML_Direct_Multiclass_LightGBM')
report_stack = get_per_class_breakdown(y_test, preds_test_stack, probs_test_stack, 'FedMML_Hierarchical_Stacking_LightGBM')

print("=" * 110)
print("   MODEL A: DIRECT 5-CLASS MULTI-CLASS LIGHTGBM HOLDOUT TEST REPORT")
print("=" * 110)
print(report_mc.to_string(index=False))
print("=" * 110 + chr(10))

print("=" * 110)
print("   MODEL B: HIERARCHICAL LIGHTGBM STACKING PIPELINE HOLDOUT TEST REPORT")
print("=" * 110)
print(report_stack.to_string(index=False))
print("=" * 110 + chr(10))

# Consolidated Comparison Table
comp_rows = []
for i in range(len(report_mc)):
    cls_lbl = report_mc.loc[i, 'Class']
    b_mc, b_st = report_mc.loc[i, 'Balanced_Accuracy'], report_stack.loc[i, 'Balanced_Accuracy']
    r_mc, r_st = report_mc.loc[i, 'Recall_Sensitivity'], report_stack.loc[i, 'Recall_Sensitivity']
    a_mc, a_st = report_mc.loc[i, 'ROC_AUC'], report_stack.loc[i, 'ROC_AUC']
    comp_rows.append({
        'Class': cls_lbl,
        'Multiclass_LGBM_BalAcc': f"{b_mc*100:.2f}%",
        'Stacking_LGBM_BalAcc': f"{b_st*100:.2f}%",
        'Delta_BalAcc': f"{(b_st - b_mc)*100:+.2f}%",
        'Multiclass_LGBM_Recall': f"{r_mc*100:.2f}%",
        'Stacking_LGBM_Recall': f"{r_st*100:.2f}%",
        'Multiclass_LGBM_AUC': round(a_mc, 4),
        'Stacking_LGBM_AUC': round(a_st, 4)
    })

comp_df = pd.DataFrame(comp_rows)
print("=" * 105)
print("     CONSOLIDATED MODEL COMPARISON: DIRECT MULTICLASS vs HIERARCHICAL STACKING")
print("=" * 105)
print(comp_df.to_string(index=False))
print("=" * 105 + chr(10))

# Export CSV Reports
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)

report_mc.to_csv(os.path.join(reports_dir, 'fedmml_direct_multiclass_lightgbm_report.csv'), index=False)
report_stack.to_csv(os.path.join(reports_dir, 'fedmml_hierarchical_stacking_lightgbm_report.csv'), index=False)
comp_df.to_csv(os.path.join(reports_dir, 'fedmml_lightgbm_model_comparison.csv'), index=False)
print(f"✓ Reports exported successfully to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 6: Diagnostic Visualizations (Confusion Matrices, ROC Curves & Feature Importance)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. Confusion Matrices
cm_mc      = confusion_matrix(y_test, preds_test_mc, labels=[1, 2, 3, 4, 5])
cm_mc_norm = cm_mc.astype('float') / cm_mc.sum(axis=1)[:, np.newaxis]

cm_st      = confusion_matrix(y_test, preds_test_stack, labels=[1, 2, 3, 4, 5])
cm_st_norm = cm_st.astype('float') / cm_st.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

annot_mc = np.empty_like(cm_mc, dtype=object)
for i in range(5):
    for j in range(5):
        annot_mc[i, j] = f"{cm_mc[i, j]:,}\n({cm_mc_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_mc_norm, annot=annot_mc, fmt='', cmap='Blues', cbar=True, ax=axes[0],
            vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels)
axes[0].set_title(
    f"Model A: Direct 5-Class LightGBM (FedMML Dataset)\n"
    f"Macro Balanced Acc: {report_mc.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro AUC: {report_mc.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

annot_st = np.empty_like(cm_st, dtype=object)
for i in range(5):
    for j in range(5):
        annot_st[i, j] = f"{cm_st[i, j]:,}\n({cm_st_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_st_norm, annot=annot_st, fmt='', cmap='Greens', cbar=True, ax=axes[1],
            vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels)
axes[1].set_title(
    f"Model B: Hierarchical Stacking LightGBM (FedMML Dataset)\n"
    f"Macro Balanced Acc: {report_stack.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro AUC: {report_stack.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[1].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, "fedmml_lightgbm_confusion_matrix.png")
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix comparison saved to: {cm_plot_path}")

# 2. Multiclass ROC-AUC Curves
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

def plot_roc_on_ax(ax, probs, title_text):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_on_ax(axes[0], probs_test_mc, f"Model A: Direct Multiclass LightGBM ROC (Macro AUC = {report_mc.loc[5, 'ROC_AUC']:.4f})")
plot_roc_on_ax(axes[1], probs_test_stack, f"Model B: Hierarchical Stacking LightGBM ROC (Macro AUC = {report_stack.loc[5, 'ROC_AUC']:.4f})")

plt.tight_layout()
roc_plot_path = os.path.join(plots_dir, "fedmml_lightgbm_roc_auc_curve.png")
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ ROC-AUC curves comparison saved to: {roc_plot_path}")

# 3. Feature Importance Visualization
fig, ax = plt.subplots(figsize=(10, 5.5))
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Gain': model_mc_lgbm.booster_.feature_importance(importance_type='gain'),
    'Importance_Split': model_mc_lgbm.booster_.feature_importance(importance_type='split')
}).sort_values('Importance_Gain', ascending=False)

sns.barplot(data=feat_imp, x='Importance_Gain', y='Feature', palette='Blues_r', ax=ax, edgecolor='black')
ax.set_title('LightGBM Feature Importance (Information Gain on FedMML Triage Features)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Total Information Gain', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')

for p in ax.patches:
    ax.annotate(f"{p.get_width():,.1f}", (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9.5, fontweight='bold', xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
feat_imp_path = os.path.join(plots_dir, "fedmml_lightgbm_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Importance plot saved to: {feat_imp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'scaler': scaler,
    'feature_names': feature_names,
    'model_multiclass_lgbm': model_mc_lgbm,
    'hierarchical_stacking': {
        'l1_model': l1_model,
        'l2_model': l2_model,
        'l3a_model': l3a_model,
        'l3b_model': l3b_model,
        'meta_learner': meta_learner
    }
}

bundle_file = os.path.join(deploy_dir, 'fedmml_lightgbm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_LightGBM_ED_Triage_Classifier',
    dataset='datasets/fedmml_ed_triage_dataset.csv',
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    target='esi_level',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    total_complete_cases=len(clean_df),
    train_samples=len(X_train),
    val_samples=len(X_val),
    test_samples=len(X_test),
    direct_multiclass_metrics=report_mc.to_dict(orient='records'),
    hierarchical_stacking_metrics=report_stack.to_dict(orient='records'),
    comparison_metrics=comp_rows
)

manifest_file = os.path.join(deploy_dir, 'fedmml_lightgbm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ FedMML Production Bundle   : {bundle_file}")
print(f"✓ FedMML Production Manifest : {manifest_file}")